# RAGAS Automated Evaluation Analysis

Reference-free evaluation of 46 questions across 6 LLM × embedding combinations using Claude Haiku as judge.

**Metrics (0.0 – 1.0, higher is better)**
- **Context Relevance** : is the retrieved context useful for the question?
- **Faithfulness** : is the answer grounded in the retrieved context?
- **Answer Relevancy** : does the answer address the question?

Difficulty tiers: Easy (Q1–Q17), Medium (Q18–Q31), Hard (Q32–Q46)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20,
})

raw = pd.read_csv('ragas_raw.csv')
summary = pd.read_csv('ragas_summary.csv', index_col=0)

METRICS = ['context_relevancy', 'faithfulness', 'answer_relevancy']
METRIC_LABELS = ['Context Relevance', 'Faithfulness', 'Answer Relevancy']

MODELS = list(summary.index)
short = [m.replace(' + ', '\n') for m in MODELS]

print('Raw shape:', raw.shape)
print('Combos:', MODELS)
print(summary.round(3))

## Summary Table

In [ ]:
display(summary.rename(columns={
    'context_relevancy': 'Context Relevance',
    'faithfulness': 'Faithfulness',
    'answer_relevancy': 'Answer Relevancy'
}).style.format('{:.3f}').background_gradient(cmap='RdYlGn', axis=None, vmin=0.5, vmax=1.0))

## Overall Scores per Combination

In [ ]:
COLORS = ['#0072B2', '#56B4E9', '#009E73', '#E69F00', '#F0E442', '#D55E00']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, label in zip(axes, METRICS, METRIC_LABELS):
    vals = summary[metric].values
    bars = ax.barh(range(len(MODELS)), vals, color=COLORS, edgecolor='black', linewidth=0.5, height=0.6)
    for bar, val in zip(bars, vals):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=11, fontweight='bold')
    ax.set_yticks(range(len(MODELS)))
    ax.set_yticklabels(short, fontsize=11)
    ax.set_xlabel('Score', fontsize=12)
    ax.set_xlim(0, 1.18)
    ax.set_title(label, fontsize=13, pad=8)
    ax.axvline(0.8, color='gray', linestyle='--', linewidth=0.8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('RAGAS Scores by Model Combination', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('ragas_overall.png', dpi=200, bbox_inches='tight')
plt.show()

## Scores by Difficulty Tier

In [ ]:
tier_data = raw.groupby(['combo', 'tier'])[METRICS].mean().unstack('tier')

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
TIER_COLORS = {'Easy': '#009E73', 'Medium': '#56B4E9', 'Hard': '#D55E00'}
x = np.arange(len(MODELS))
w = 0.25

for ax, metric, label in zip(axes, METRICS, METRIC_LABELS):
    for i, (tier, color) in enumerate(TIER_COLORS.items()):
        vals = [tier_data[metric].loc[m, tier] if tier in tier_data[metric].columns else 0
                for m in MODELS]
        bars = ax.bar(x + i*w, vals, w, label=tier, color=color, edgecolor='black', linewidth=0.4)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x + w)
    ax.set_xticklabels(short, fontsize=9)
    ax.set_ylim(0, 1.22)
    ax.set_title(label, fontsize=13)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if ax == axes[0]:
        ax.set_ylabel('Mean Score', fontsize=12)
        ax.legend(fontsize=11, framealpha=0.7)

fig.suptitle('RAGAS Scores by Difficulty Tier', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('ragas_by_tier.png', dpi=200, bbox_inches='tight')
plt.show()

## Heatmaps: Per-Question Scores

In [ ]:
cmap = mcolors.LinearSegmentedColormap.from_list('ragas', ['#D55E00', '#F0E442', '#009E73'])

fig, axes = plt.subplots(1, 3, figsize=(20, 15))

for ax, metric, label in zip(axes, METRICS, METRIC_LABELS):
    pivot = raw.pivot_table(index='question', columns='combo', values=metric, aggfunc='mean')
    pivot = pivot.reindex(columns=MODELS)
    questions = raw['question'].unique()
    pivot = pivot.reindex(questions)

    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap, vmin=0, vmax=1)

    short_cols = [m.replace(' + ', '\n') for m in MODELS]
    ax.set_xticks(range(len(MODELS)))
    ax.set_xticklabels(short_cols, fontsize=13)
    ax.set_yticks(range(len(questions)))
    ax.set_yticklabels([f'Q{i+1}' for i in range(len(questions))],
                       fontsize=11 if ax == axes[0] else 0)

    for sep in [16.5, 30.5]:
        ax.axhline(sep, color='black', linewidth=2)

    ax.set_title(label, fontsize=16, pad=10)
    cb = plt.colorbar(im, ax=ax, shrink=0.4)
    cb.ax.tick_params(labelsize=12)
    cb.set_label('Score', fontsize=13)

    n_cols = len(MODELS)
    for tier_name, start, end in [('Easy', 0, 16), ('Medium', 17, 30), ('Hard', 31, 45)]:
        mid = (start + end) / 2
        ax.text(n_cols + 0.1, mid, tier_name, ha='left', va='center',
                fontsize=14, fontweight='bold', rotation=90)
    ax.set_xlim(-0.5, n_cols + 0.6)

fig.suptitle('Per-Question RAGAS Scores by Model', fontsize=18, y=1.01)
plt.tight_layout()
plt.savefig('ragas_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

## Claude vs Ollama Summary

In [ ]:
claude_rows = summary[summary.index.str.startswith('claude')]
ollama_rows = summary[summary.index.str.startswith('ollama')]

print('Claude averages:')
print(claude_rows[METRICS].mean().round(3).to_string())
print()
print('Ollama averages:')
print(ollama_rows[METRICS].mean().round(3).to_string())
print()
print('Per embedding comparison:')
for emb in ['ollama', 'sentence', 'openai']:
    c = summary.loc[f'claude + {emb}']
    o = summary.loc[f'ollama + {emb}']
    print(f'  {emb:<10}', end='')
    for m in METRICS:
        print(f'  {m[:4]}  C={c[m]:.3f} O={o[m]:.3f} diff={c[m]-o[m]:+.3f}', end='')
    print()

## Score Distribution (Boxplot)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for ax, metric, label in zip(axes, METRICS, METRIC_LABELS):
    data = [raw[raw['combo'] == m][metric].dropna().values for m in MODELS]
    bp = ax.boxplot(data, patch_artist=True, medianprops={'color': 'black', 'linewidth': 2})
    for patch, color in zip(bp['boxes'], COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax.set_xticks(range(1, len(MODELS)+1))
    ax.set_xticklabels(short, fontsize=12)
    ax.set_title(label, fontsize=16)
    ax.set_ylim(-0.05, 1.1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if ax == axes[0]:
        ax.set_ylabel('Score', fontsize=14)

fig.suptitle('Score Distribution per Model Combination', fontsize=18, y=1.02)
plt.tight_layout()
plt.savefig('ragas_boxplot.png', dpi=200, bbox_inches='tight')
plt.show()